# Notebook 13: The Agent2Agent Protocol (A2A)

## Letting AI Agents Talk to Each Other, the Standard Way

In Notebook 12 you learned MCP — the standard for connecting **agents to tools**. But what
happens when the thing on the other end isn't a tool, it's *another agent* — built by another
team, on another framework, in another company? That's what **A2A (Agent2Agent)** is for.

In this notebook you will learn, from zero:

1. **Why A2A exists** — the agent-silos problem, and why agents are not tools
2. **The core concepts** — Agent Cards, Tasks, Messages, Parts, Artifacts
3. **The task lifecycle** — including multi-turn `input-required` conversations
4. **Building an A2A server from a LangGraph agent** with the official `a2a-sdk`
5. **Streaming and the wire format** — what actually travels over HTTP

**Prerequisites**: Notebooks 01-12, `OPENAI_API_KEY` in `.env`
**Time**: 45-55 minutes

---
# Part 0: Why Does A2A Exist?

## The Agent-Silos Problem

Ask an AI assistant to *"plan an international trip"* and it needs a flight-booking agent, a
hotel agent, a currency agent, and a local-tours agent — likely built by **different teams, on
different frameworks** (LangGraph, CrewAI, Google ADK...). Without a standard:

- every integration is **custom point-to-point code** (the N×M problem again, one level up)
- agents get **wrapped as tools**, which strips away what makes them agents — they can't
  negotiate, ask clarifying questions, or manage long-running work through a tool interface
- there's **no consistent security model** between organizations

**A2A solves this** by exposing agents *as agents*: any A2A client can discover any A2A
server's capabilities and collaborate with it, whatever framework either side uses.

## Where A2A Sits in the Stack

| Layer | Question it answers | Example |
|---|---|---|
| **Models** | How does the agent reason? | GPT, Gemini, Claude |
| **Frameworks** | How is the agent built? | LangGraph, CrewAI, ADK |
| **MCP** | How does an agent reach tools & data? | Your ticket server (Notebook 12) |
| **A2A** | How do agents reach *each other*? | This notebook |

> **MCP stays within one agent** (agent → tools). **A2A crosses the boundary between agents.**
> They are complementary — you'll see them combined at the end.

## Key Benefits

- **Interoperability** — agents from different vendors/frameworks collaborate
- **Autonomy & opacity** — each agent keeps its internal state, memory, and tools private
- **No custom integrations** — one protocol reaches agents anywhere (local, cloud, vendor-hosted)
- **Long-running operations** — streaming (SSE) and push notifications are built in
- **Enterprise-ready** — standard web security: HTTPS, auth schemes declared up front

In [1]:
import os
import threading
import time
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

from importlib.metadata import version
print("=== Environment ===")
for pkg in ["a2a-sdk", "langchain", "langgraph"]:
    print(f"  {pkg:<10} {version(pkg)}")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print(f"LLM: {MODEL}")

A2A_PORT = 9999
A2A_BASE_URL = f"http://127.0.0.1:{A2A_PORT}"

=== Environment ===
  a2a-sdk    1.3.1
  langchain  1.5.2
  langgraph  1.2.24
LLM: gpt-4o-mini


---
# Part 1: A2A Fundamentals

## The Actors

- **User** — a human or automated service with a goal
- **A2A Client** — the agent/application acting for the user (initiates communication)
- **A2A Server** — a remote agent exposing an HTTP endpoint that speaks A2A. To the client it
  is **opaque**: its internals, memory, and tools stay hidden

## The Core Objects

| Object | What it is |
|---|---|
| **Agent Card** | A JSON "business card" served at `/.well-known/agent-card.json`: identity, endpoint URL, skills, capabilities, auth requirements. *Discovery starts here.* |
| **Task** | A stateful unit of work with a unique `id` and a lifecycle. Long-running work happens inside tasks |
| **Message** | One turn of communication (role `user` or `agent`), containing Parts |
| **Part** | The content container: exactly one of `text`, `raw` (bytes), `url`, or `data` (JSON) |
| **Artifact** | A tangible output of a task (a document, an image, a result) — also made of Parts |
| **contextId** | Groups related tasks/messages into one conversation |

## The Task Lifecycle

```
submitted ──► working ──► completed        (terminal)
                 │
                 ├──► input-required ──► (client replies) ──► working ...
                 ├──► auth-required
                 ├──► failed               (terminal)
                 ├──► canceled             (terminal)
                 └──► rejected             (terminal)
```

**Terminal tasks are immutable** — a refinement creates a *new* task in the same `contextId`.

## Three Interaction Mechanisms

1. **Request/response + polling** — send a message, poll `GetTask` for updates
2. **Streaming (SSE)** — `SendStreamingMessage` keeps the connection open; the server pushes
   `task`, `status_update`, and `artifact_update` events in real time
3. **Push notifications** — the server calls a client-provided webhook (for very long tasks)

All bindings use standard web tech: JSON-RPC 2.0 over HTTP(S) (+SSE), gRPC, or REST.

## Our Agent: A LangGraph Currency Converter

We'll build a small but real LangGraph agent — a ReAct agent with an exchange-rate tool —
and then expose it as an A2A server. This mirrors the official A2A LangGraph sample, with a
deterministic rate table so results are reproducible in class.

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool

RATES = {  # static table: 1 unit of base -> target
    ("USD", "EUR"): 0.92, ("USD", "GBP"): 0.79, ("USD", "INR"): 85.6, ("USD", "JPY"): 151.3,
    ("EUR", "USD"): 1.09, ("GBP", "USD"): 1.27,
}


@tool
def get_exchange_rate(base: str, target: str) -> str:
    """Get the exchange rate from one currency code to another (e.g. USD to EUR)."""
    rate = RATES.get((base.upper(), target.upper()))
    if rate is None:
        return f"No rate available for {base}->{target}. Supported pairs involve USD, EUR, GBP."
    return f"1 {base.upper()} = {rate} {target.upper()}"


currency_agent = create_agent(
    f"openai:{MODEL}",
    [get_exchange_rate],
    system_prompt=(
        "You are a currency conversion agent. Always use the get_exchange_rate tool, "
        "compute the converted amount, and answer in one short sentence with the numbers."
    ),
)

# Local smoke test — before any protocol is involved
_smoke = currency_agent.invoke({"messages": [("user", "How much is 100 USD in EUR?")]})
print("Local agent says:", _smoke["messages"][-1].content)

Local agent says: 100 USD is 92 EUR.


## The A2A Building Blocks

Wrapping that agent as an A2A server takes four pieces (all from `a2a-sdk`):

1. **`AgentSkill`** — one capability, with tags and example prompts
2. **`AgentCard`** — the public description: name, URL, capabilities (streaming?), skills
3. **`AgentExecutor`** — *your code*: receives requests, drives the LangGraph agent, emits
   task events through a `TaskUpdater`
4. **`DefaultRequestHandler` + routes** — the SDK's plumbing: JSON-RPC routing, task store,
   agent-card endpoint. We serve it with Starlette + uvicorn

The executor below also demonstrates the **`input-required`** state: if the user hasn't said
*which* currency to convert to, the task pauses and asks — then resumes when they reply.

In [3]:
import re

from a2a.helpers import get_message_text, new_task_from_user_message, new_text_message, new_text_part
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.tasks import TaskUpdater
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill, TaskState

SUPPORTED_TARGETS = ["EUR", "GBP", "INR", "JPY"]


class CurrencyAgentExecutor(AgentExecutor):
    """Bridges the A2A protocol and our LangGraph currency agent."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # 1. Get or create the task for this request
        if context.current_task:
            task = context.current_task
        else:
            task = new_task_from_user_message(context.message)
            await event_queue.enqueue_event(task)
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.start_work()

        # 2. Combine the whole conversation so far (multi-turn aware)
        history_texts = [get_message_text(m) for m in (task.history or []) if m.role == 1]
        full_query = " ".join(history_texts + [get_message_text(context.message)]).strip()

        # 3. Multi-turn gate: do we know the target currency yet?
        has_target = any(c in full_query.upper() for c in SUPPORTED_TARGETS)
        if not has_target:
            await updater.requires_input(new_text_message(
                "Which currency should I convert to? I support: " + ", ".join(SUPPORTED_TARGETS)
            ))
            return

        # 4. Run the LangGraph agent and publish its answer as an artifact
        result = await currency_agent.ainvoke({"messages": [("user", full_query)]})
        answer = result["messages"][-1].content
        await updater.add_artifact(parts=[new_text_part(text=answer, media_type="text/plain")],
                                   name="conversion_result")
        await updater.complete(new_text_message("Conversion completed."))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("This demo agent does not support cancellation.")


# ── The Agent Card: what the world sees ──────────────────────────────────────
currency_skill = AgentSkill(
    id="currency_conversion",
    name="Currency Conversion",
    description="Converts amounts between currencies using live rates (USD, EUR, GBP, INR, JPY).",
    tags=["finance", "currency", "conversion"],
    examples=["How much is 100 USD in EUR?", "Convert 50 dollars to yen"],
)

agent_card = AgentCard(
    name="Currency Converter Agent",
    description="A LangGraph agent that converts currency amounts, exposed over A2A.",
    version="1.0.0",
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    capabilities=AgentCapabilities(streaming=True),
    supported_interfaces=[
        AgentInterface(protocol_binding="JSONRPC", url=f"{A2A_BASE_URL}/", protocol_version="1.0")
    ],
    skills=[currency_skill],
)

print("Executor + Agent Card defined.")
print(f"Card: '{agent_card.name}' v{agent_card.version}, streaming={agent_card.capabilities.streaming}")

Executor + Agent Card defined.
Card: 'Currency Converter Agent' v1.0.0, streaming=True


## Serving the Agent

The SDK's route factories wire everything into a standard ASGI app:

- `create_agent_card_routes(card)` → serves `/.well-known/agent-card.json`
- `create_jsonrpc_routes(handler, "/")` → handles all A2A JSON-RPC calls

We run uvicorn in a **background thread** so the notebook keeps executing. In production
this would simply be your server's main process.

In [4]:
import httpx
import uvicorn
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from starlette.applications import Starlette

request_handler = DefaultRequestHandler(
    agent_executor=CurrencyAgentExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=agent_card,
)

routes = []
routes.extend(create_agent_card_routes(agent_card))
routes.extend(create_jsonrpc_routes(request_handler, "/"))
a2a_app = Starlette(routes=routes)

uvicorn_config = uvicorn.Config(a2a_app, host="127.0.0.1", port=A2A_PORT, log_level="error")
uvicorn_server = uvicorn.Server(uvicorn_config)
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()

# Wait until the agent card endpoint answers
for attempt in range(40):
    try:
        r = httpx.get(f"{A2A_BASE_URL}/.well-known/agent-card.json", timeout=2)
        if r.status_code == 200:
            break
    except Exception:
        time.sleep(0.5)

print(f"✅ A2A server running on {A2A_BASE_URL}")

✅ A2A server running on http://127.0.0.1:9999


## Discovery: Fetching the Agent Card

Before talking to any A2A agent, a client fetches its card. This is the *entire* integration
surface: no SDKs shared, no code exchanged — just this JSON document.

In [5]:
card_json = httpx.get(f"{A2A_BASE_URL}/.well-known/agent-card.json").json()

import json
print(json.dumps(card_json, indent=2)[:1200])

{
  "name": "Currency Converter Agent",
  "description": "A LangGraph agent that converts currency amounts, exposed over A2A.",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:9999/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "currency_conversion",
      "name": "Currency Conversion",
      "description": "Converts amounts between currencies using live rates (USD, EUR, GBP, INR, JPY).",
      "tags": [
        "finance",
        "currency",
        "conversion"
      ],
      "examples": [
        "How much is 100 USD in EUR?",
        "Convert 50 dollars to yen"
      ]
    }
  ]
}


## The Wire Format: Raw JSON-RPC

Under the hood, A2A is JSON-RPC 2.0 over HTTP. Let's send a `SendMessage` request **by hand**
with `httpx` — no SDK — so you can see exactly what travels on the wire. Two details to
notice: v1.0 servers require the **`A2A-Version: 1.0` HTTP header**, and the payload has the
shapes you learned earlier — a `message` with `parts`, answered by a `task` with a `status`,
`history`, and `artifacts`.

In [6]:
raw_request = {
    "jsonrpc": "2.0",
    "id": "req-001",
    "method": "SendMessage",
    "params": {
        "message": {
            "messageId": "msg-001",
            "role": "ROLE_USER",
            "parts": [{"text": "How much is 250 USD in GBP?"}],
        }
    },
}

print("── REQUEST ─────────────────────────────────")
print(json.dumps(raw_request, indent=2))

raw_response = httpx.post(
    f"{A2A_BASE_URL}/",
    json=raw_request,
    headers={"A2A-Version": "1.0"},  # v1.0 servers require the version header
    timeout=60,
).json()

print("\n── RESPONSE (trimmed) ──────────────────────")
task = raw_response["result"]["task"]
print("task id:      ", task["id"])
print("context id:   ", task["contextId"])
print("status:       ", task["status"]["state"])
for artifact in task.get("artifacts", []):
    print("artifact:     ", artifact["name"], "→", artifact["parts"][0]["text"])

── REQUEST ─────────────────────────────────
{
  "jsonrpc": "2.0",
  "id": "req-001",
  "method": "SendMessage",
  "params": {
    "message": {
      "messageId": "msg-001",
      "role": "ROLE_USER",
      "parts": [
        {
          "text": "How much is 250 USD in GBP?"
        }
      ]
    }
  }
}



── RESPONSE (trimmed) ──────────────────────
task id:       619223b8-6b70-431d-a373-56a012022569
context id:    d27ed76a-573f-4852-b4f3-ce19995ad0c3
status:        TASK_STATE_COMPLETED
artifact:      conversion_result → 250 USD is 197.5 GBP.


## The SDK Client

Writing JSON-RPC by hand is instructive but tedious. The SDK's `create_client` reads the
Agent Card and gives you a typed client. Non-streaming clients yield the final result;
streaming clients yield events as they happen.

In [7]:
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Role, SendMessageRequest

# 1. Resolve the card (same JSON we fetched above, now typed)
async with httpx.AsyncClient() as httpx_client:
    resolver = A2ACardResolver(httpx_client=httpx_client, base_url=A2A_BASE_URL)
    resolved_card = await resolver.get_agent_card()

# 2. Create a non-streaming client from the card
client = await create_client(agent=resolved_card, client_config=ClientConfig(streaming=False))

# 3. Send a message and collect the final task
request = SendMessageRequest(
    message=new_text_message("How much is 100 USD in INR?", role=Role.ROLE_USER)
)
final_task = None
async for response in client.send_message(request):
    final_task = response.task  # StreamResponse carries task | message | status_update | artifact_update

print("status:  ", TaskState.Name(final_task.status.state))
print("artifact:", final_task.artifacts[0].parts[0].text)

status:   TASK_STATE_COMPLETED
artifact: 100 USD is 8560 INR.


## Multi-Turn: The `input-required` Dance

Ask an incomplete question and the agent pauses the task with `input-required`. The client
replies **in the same `contextId` and `taskId`** — the executor sees the task's history and
finishes the job. This is what "agents, not tools" looks like in practice: the remote agent
*negotiates*.

In [8]:
# Turn 1 — no target currency
request1 = SendMessageRequest(message=new_text_message("How much is 500 USD?", role=Role.ROLE_USER))
task1 = None
async for response in client.send_message(request1):
    task1 = response.task

print("Turn 1 status:", TaskState.Name(task1.status.state))  # → TASK_STATE_INPUT_REQUIRED
print("Agent asks:   ", task1.status.message.parts[0].text)

# Turn 2 — answer the question, referencing the same task & context
request2 = SendMessageRequest(
    message=new_text_message(
        "JPY",
        role=Role.ROLE_USER,
        context_id=task1.context_id,
        task_id=task1.id,
    )
)
task2 = None
async for response in client.send_message(request2):
    task2 = response.task

print("\nTurn 2 status:", TaskState.Name(task2.status.state))
print("Final answer: ", task2.artifacts[0].parts[0].text)

Turn 1 status: TASK_STATE_INPUT_REQUIRED
Agent asks:    Which currency should I convert to? I support: EUR, GBP, INR, JPY



Turn 2 status: TASK_STATE_COMPLETED
Final answer:  500 USD is 75,650 JPY.


## Streaming: Watching the Task Work

With `ClientConfig(streaming=True)`, `send_message` becomes a live event feed over SSE.
You receive the same event types the executor emits: the initial `task`, then
`status_update` events (working → completed) and `artifact_update` events (the result).

In [9]:
streaming_client = await create_client(agent=resolved_card, client_config=ClientConfig(streaming=True))

request3 = SendMessageRequest(
    message=new_text_message("Convert 42 USD to EUR please", role=Role.ROLE_USER)
)

print("── live event stream ───────────────────────")
async for event in streaming_client.send_message(request3):
    kind = event.WhichOneof("payload")
    if kind == "task":
        print(f"[task]            id={event.task.id[:8]}… state={TaskState.Name(event.task.status.state)}")
    elif kind == "status_update":
        state = TaskState.Name(event.status_update.status.state)
        msg = ""
        if event.status_update.status.HasField("message"):
            msg = " — " + event.status_update.status.message.parts[0].text
        print(f"[status_update]   {state}{msg}")
    elif kind == "artifact_update":
        text = event.artifact_update.artifact.parts[0].text
        print(f"[artifact_update] {event.artifact_update.artifact.name}: {text}")

await streaming_client.close()

── live event stream ───────────────────────
[task]            id=489dfaf0… state=TASK_STATE_SUBMITTED
[status_update]   TASK_STATE_WORKING


[artifact_update] conversion_result: 42 USD is 38.64 EUR.
[status_update]   TASK_STATE_COMPLETED — Conversion completed.


## Push Notifications (Concept)

For very long tasks, even an SSE stream may outlive the client's patience. A2A's answer:
**push notifications**. The client registers a webhook URL per task
(`CreateTaskPushNotificationConfig`), disconnects, and the server POSTs task updates to that
URL when they happen. The `DefaultRequestHandler` supports this with a
`push_config_store` + `push_sender` — we skip the live demo (it needs a publicly reachable
webhook), but the pieces are: declare `push_notifications=True` in the card's capabilities,
register the config, receive signed callbacks.

---
# Part 3: A2A + MCP Together, and Production Notes

## The Full Picture

The two protocols compose naturally:

```
                    ┌─────────────────────────────┐
   other agents ───►│  A2A endpoint (this nb)     │
   (any framework)  │  ┌───────────────────────┐  │
                    │  │ LangGraph agent        │  │
                    │  │  ├─ reasoning (LLM)    │  │
                    │  │  └─ tools ──► MCP ────►│──┼──► ticket server, DBs, APIs
                    │  └───────────────────────┘  │      (Notebook 12)
                    └─────────────────────────────┘
```

- **MCP** answers: *how does my agent use tools and data?*
- **A2A** answers: *how does my agent collaborate with other agents?*

A production system typically uses both: agents are built with a framework (LangGraph),
equipped with tools via MCP, and exposed to other agents via A2A.

## Production Notes

- **Treat remote agents as untrusted.** An Agent Card is external input — sanitize names,
  descriptions, and skill text before putting them into an LLM prompt (prompt-injection risk).
- **Auth lives in the card.** Declare security schemes in the Agent Card; send credentials in
  HTTP headers, never inside A2A message payloads. Use HTTPS in production.
- **Opacity is a feature.** Never expose internal memory or tool traces through A2A — share
  only Messages and Artifacts.
- **Terminal tasks are immutable.** Refinements are new tasks in the same `contextId`.

---
## Summary

| Concept | What it is | Where you saw it |
|---|---|---|
| Agent Card | Discovery document at `/.well-known/agent-card.json` | Fetched live over HTTP |
| Task & lifecycle | Stateful unit of work; terminal states are immutable | `working` → `completed`, `input-required` |
| Message / Part / Artifact | Turns, content containers, tangible outputs | Executor + client cells |
| contextId | Groups related tasks into one conversation | Multi-turn example |
| AgentExecutor / TaskUpdater | Your logic ↔ the protocol | `CurrencyAgentExecutor` |
| JSON-RPC binding | The wire format | Raw `httpx` POST |
| SSE streaming | Live task/status/artifact events | Streaming client |
| Push notifications | Webhooks for long-running tasks | Concept section |

**Links**

- [A2A Documentation](https://a2a-protocol.org/latest/)
- [Protocol specification](https://a2a-protocol.org/latest/specification/)
- [Python SDK quickstart](https://a2a-protocol.org/latest/tutorials/python/1-introduction/)
- [Official samples (incl. LangGraph)](https://github.com/a2aproject/a2a-samples)
- [a2a-cli](https://github.com/a2aproject/a2a-cli)

**Series complete** — from StateGraph fundamentals (Notebook 01) to agents that use tools
(MCP) and talk to each other (A2A). 🎓

In [10]:
# Shut down the A2A server cleanly
uvicorn_server.should_exit = True
time.sleep(1)
print("✅ A2A server stopped. Notebook complete.")

✅ A2A server stopped. Notebook complete.
